In [8]:
from importlib import import_module
from pathlib import Path
import json
import random
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / '.data/extracted').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

evaluation = import_module('src.steps.03_evaluation')
EvaluationItem, Evaluator = evaluation.EvaluationItem, evaluation.Evaluator
SAMPLE_SIZE, SEED, WORKERS = 15, 42, 2

In [9]:
def load_items(limit=1, seed=42):
    extracted = PROJECT_ROOT / '.data/extracted'
    annotated = PROJECT_ROOT / '.data/annotated'
    texts = {p.relative_to(extracted).with_suffix('').as_posix(): p for p in extracted.rglob('*.txt')}
    labels = {p.relative_to(annotated).with_suffix('').as_posix(): p for p in annotated.rglob('*.json')}
    pairs = sorted(texts.keys() & labels.keys())
    if not pairs:
        raise FileNotFoundError('No matching resume text/JSON pairs found.')
    if limit is not None:
        pairs = random.Random(seed).sample(pairs, min(limit, len(pairs)))

    items = []
    for pair in pairs:
        target = json.loads(labels[pair].read_text(encoding='utf-8'))
        items.append(EvaluationItem(
            resume_id=Path(pair).stem,
            category=Path(pair).parent.as_posix(),
            resume_text=texts[pair].read_text(encoding='utf-8'),
            target_json=target,
        ))
    return items, len(texts.keys() & labels.keys())

In [10]:
items, available_pairs = load_items(SAMPLE_SIZE, SEED)
print(f'Available paired resumes: {available_pairs:,}')
print(f'Loaded for this test: {len(items)}')

Available paired resumes: 13,591
Loaded for this test: 15


In [11]:
def generate_response(item):
    return item.target_json  

In [12]:


results = Evaluator([items[2]]).evaluate(
    callback=generate_response,
    workers=1,
)
result = results.results[0]
print('Status:', result.status)
print('Jev score:', result.judge_score)
print('Question breakdown:', json.dumps(result.score_breakdown, indent=2))

Evaluating with Jev (1 workers):   0%|          | 0/1 [00:00<?, ?resume/s]


Jev question breakdown — resume resume_f1c87e0958452b7d
- personal_information_full_name_match [personal_information.full_name]: Jev raw=0.990; after thresholds=1.000 (zero≤20%, full≥80%); weight=1; weighted contribution=1.000
  expected: "Jessica Claire"
  generated: "Jessica Claire"
- personal_information_professional_title_match [personal_information.professional_title]: Jev raw=0.980; after thresholds=1.000 (zero≤20%, full≥80%); weight=1; weighted contribution=1.000
  expected: null
  generated: null
- personal_information_email_match [personal_information.email]: Jev raw=1.000; after thresholds=1.000 (zero≤20%, full≥80%); weight=1; weighted contribution=1.000
  expected: "resumesample@example.com"
  generated: "resumesample@example.com"
- personal_information_address_match [personal_information.address]: Jev raw=0.990; after thresholds=1.000 (zero≤20%, full≥80%); weight=1; weighted contribution=1.000
  expected: "100 Montgomery St. 10th Floor"
  generated: "100 Montgomery St. 10t

Evaluating with Jev (1 workers): 100%|██████████| 1/1 [00:01<00:00,  1.30s/resume]

Status: SCORED
Jev score: 1.0
Question breakdown: [
  {
    "name": "personal_information_full_name_match",
    "path": "personal_information.full_name",
    "type": "noul",
    "weight": 1.0,
    "value": 1.0,
    "raw_value": 0.99,
    "thresholds": {
      "zero_at_or_below": 0.2,
      "full_at_or_above": 0.8
    },
    "calibrated": true,
    "applicable": true,
    "answer": 0.99,
    "confidence": null,
    "probabilities": {
      "yes": 0.99,
      "no": 0.010000000000000009
    },
    "weighted_contribution": 1.0,
    "expected_value": "Jessica Claire",
    "generated_value": "Jessica Claire"
  },
  {
    "name": "personal_information_professional_title_match",
    "path": "personal_information.professional_title",
    "type": "noul",
    "weight": 1.0,
    "value": 1.0,
    "raw_value": 0.98,
    "thresholds": {
      "zero_at_or_below": 0.2,
      "full_at_or_above": 0.8
    },
    "calibrated": true,
    "applicable": true,
    "answer": 0.98,
    "confidence": null,
    "

## Evaluate your model

Replace `your_model.generate(...)` with your inference call, then run this cell. Set `SAMPLE_SIZE` above to the number of pairs you want to test.

In [ ]:
def model_callback(item):
    prompt = evaluation.render_json_prompt(item.resume_text)
    return your_model.generate(prompt)

# report = Evaluator(items).evaluate(model_callback, workers=WORKERS)
# print(json.dumps(report.metrics, indent=2))